In [ ]:
!pip uninstall -y protobuf mediapipe tensorflow keras

In [ ]:
!pip install protobuf==4.25.3
!pip install mediapipe==0.10.14
!pip install tensorflow==2.19.1
!pip install keras==3.5.0
!pip install scikit-learn matplotlib opencv-python kagglehub

In [ ]:
import os
import cv2
import pickle
import numpy as np
import mediapipe as mp
import matplotlib.pyplot as plt
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
from google.colab import files

files.upload()

Saving kaggle (3).json to kaggle (3).json


{'kaggle (3).json': b'{"username":"saudtechnicaltv","key":"b17d05a95c5608ef90163e5c60f1ddc9"}'}

In [ ]:
!mkdir -p ~/.kaggle

!cp kaggle.json ~/.kaggle/

!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d grassknoted/asl-alphabet

Dataset URL: https://www.kaggle.com/datasets/grassknoted/asl-alphabet
License(s): GPL-2.0
100% 1.03G/1.03G [00:59<00:00, 18.6MB/s]



In [ ]:
import zipfile

zip_path = (
    "/content/asl-alphabet.zip"
)

extract_path = (
    "/content/asl_dataset"
)

with zipfile.ZipFile(
    zip_path,
    'r'
) as zip_ref:

    zip_ref.extractall(
        extract_path
    )

print(
    "Dataset Extracted Successfully"
)

Dataset Extracted Successfully


In [ ]:
import os

dataset_path = (
    "/content/asl_dataset/"
    "asl_alphabet_train/"
    "asl_alphabet_train"
)

print(
    os.listdir(
        dataset_path
    )[:10]
)

['Q', 'nothing', 'N', 'G', 'P', 'D', 'W', 'B', 'C', 'A']


In [ ]:
import mediapipe as mp

mp_hands = mp.solutions.hands

hands = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.2,
    min_tracking_confidence=0.2
)

In [ ]:
def preprocess_image(image):

    image = cv2.resize(
        image,
        (256, 256)
    )

    image_rgb = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )

    # Brightness improve
    image_rgb = cv2.convertScaleAbs(
        image_rgb,
        alpha=1.3,
        beta=20
    )

    # Blur remove noise
    image_rgb = cv2.GaussianBlur(
        image_rgb,
        (3, 3),
        0
    )

    return image_rgb

In [ ]:
def augment_image(image):

    augmented_images = []

    augmented_images.append(image)

    # Flip
    flip = cv2.flip(image, 1)
    augmented_images.append(flip)

    rows, cols = image.shape[:2]

    # Rotate Left
    M1 = cv2.getRotationMatrix2D(
        (cols/2, rows/2),
        -10,
        1
    )

    left = cv2.warpAffine(
        image,
        M1,
        (cols, rows)
    )

    augmented_images.append(left)

    # Rotate Right
    M2 = cv2.getRotationMatrix2D(
        (cols/2, rows/2),
        10,
        1
    )

    right = cv2.warpAffine(
        image,
        M2,
        (cols, rows)
    )

    augmented_images.append(right)

    # Brightness
    bright = cv2.convertScaleAbs(
        image,
        alpha=1.2,
        beta=15
    )

    augmented_images.append(bright)

    return augmented_images

In [ ]:
all_images = []
all_labels = []

image_size = 128

max_images = 50000000
count = 0

for label in os.listdir(dataset_path):

    folder_path = os.path.join(
        dataset_path,
        label
    )

    if os.path.isdir(folder_path):

        image_files = os.listdir(
            folder_path
        )

        for image_name in image_files:

            # Stop at 10k images
            if count >= max_images:
                break

            image_path = os.path.join(
                folder_path,
                image_name
            )

            image = cv2.imread(
                image_path
            )

            if image is None:
                continue

            image = cv2.resize(
                image,
                (image_size, image_size)
            )

            augmented_images = augment_image(
                image
            )

            for img in augmented_images:

                if count >= max_images:
                    break

                all_images.append(
                    img
                )

                all_labels.append(
                    label
                )

                count += 1

print(
    "Dataset Loaded"
)

print(
    "Total Images:",
    len(all_images)
)

In [ ]:
def extract_landmarks(image):

    image = preprocess_image(image)

    results = hands.process(image)

    if results.multi_hand_landmarks:

        landmarks = []

        for hand_landmarks in results.multi_hand_landmarks:

            for lm in hand_landmarks.landmark:

                landmarks.extend([
                    lm.x,
                    lm.y,
                    lm.z
                ])

        return landmarks

    return None

In [ ]:
X = []
y = []

for image, label in zip(
    all_images,
    all_labels
):

    landmarks = extract_landmarks(image)

    if landmarks is not None:

        X.append(landmarks)
        y.append(label)

print("MediaPipe Extraction Complete")
print("Total Features:", len(X))

MediaPipe Extraction Complete
Total Features: 22826


In [ ]:
print(
    "Total Images:",
    len(all_images)
)

print(
    "Extracted Features:",
    len(X)
)

success_rate = (
    len(X)
    /
    len(all_images)
) * 100

print(
    "Detection Rate:",
    success_rate
)

Total Images: 50000
Extracted Features: 22826
Detection Rate: 45.652


In [ ]:
X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

(22826, 63)
(22826,)


In [ ]:
X = np.array(X)

X_mean = np.mean(X)
X_std = np.std(X)

X = (X - X_mean) / X_std

In [ ]:
encoder = LabelEncoder()

y_encoded = encoder.fit_transform(y)

y_categorical = to_categorical(
    y_encoded
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y_categorical,

    test_size=0.2,

    random_state=42,

    stratify=y_encoded
)

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)

Train Shape: (18260, 63)
Test Shape: (4566, 63)


In [ ]:
model = Sequential([

    Dense(
        512,
        activation='relu',
        input_shape=(63,)
    ),

    Dropout(0.6),

    Dense(
        256,
        activation='relu'
    ),

    Dropout(0.5),

    Dense(
        128,
        activation='relu'
    ),

    Dropout(0.4),

    Dense(
        64,
        activation='relu'
    ),

    Dense(
        len(np.unique(y)),
        activation='softmax'
    )
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
early_stop = EarlyStopping(

    monitor='val_accuracy',

    patience=8,

    restore_best_weights=True
)

history = model.fit(

    X_train,
    y_train,

    validation_split=0.2,

    epochs=70,

    batch_size=16,

    callbacks=[early_stop]
)

Epoch 1/70
913/913 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9919 - loss: 0.0258 - val_accuracy: 0.9967 - val_loss: 0.0071
Epoch 2/70
913/913 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9941 - loss: 0.0231 - val_accuracy: 0.9981 - val_loss: 0.0062
Epoch 3/70
913/913 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9923 - loss: 0.0258 - val_accuracy: 0.9956 - val_loss: 0.0133
Epoch 4/70
913/913 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9923 - loss: 0.0222 - val_accuracy: 0.9970 - val_loss: 0.0105
Epoch 5/70
913/913 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9937 - loss: 0.0212 - val_accuracy: 0.9964 - val_loss: 0.0081
Epoch 6/70
913/913 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9954 - loss: 0.0170 - val_accuracy: 0.9973 - val_loss: 0.0073
Epoch 7/70
913/913 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9937 - loss: 0.0256 - val_accuracy: 0.9978 - val_loss: 0.0083
Epoch 8/70
913/913 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9954 - loss: 0.0131 - val_accuracy: 0.

In [ ]:
loss, accuracy = model.evaluate(
    X_test,
    y_test
)

print(
    "Test Accuracy:",
    accuracy * 100
)

143/143 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9965 - loss: 0.0120
Test Accuracy: 99.71528649330139


In [ ]:
model.save(
    "asl_model.keras"
)

print("Model Saved Successfully")

Model Saved Successfully


In [ ]:
import cv2
import pickle
import mediapipe as mp
import numpy as np
from google.colab import files
from tensorflow.keras.models import load_model

# =====================================
# Load Model
# =====================================

model = load_model(
    "/content/asl_model.keras"
)

print("Model Loaded")

# =====================================
# Load Label Encoder
# =====================================

with open(
    "/content/label_encoder.pkl",
    "rb"
) as f:

    encoder = pickle.load(f)

print("Label Encoder Loaded")

# =====================================
# Upload Image
# =====================================

uploaded = files.upload()

image_path = list(
    uploaded.keys()
)[0]

# =====================================
# Read Image
# =====================================

image = cv2.imread(
    image_path
)

# =====================================
# Better Preprocessing
# =====================================

image = cv2.resize(
    image,
    (256,256)
)

# Brightness Improve
image = cv2.convertScaleAbs(
    image,
    alpha=1.3,
    beta=20
)

# Gaussian Blur
image = cv2.GaussianBlur(
    image,
    (3,3),
    0
)

# RGB Convert
rgb = cv2.cvtColor(
    image,
    cv2.COLOR_BGR2RGB
)

# =====================================
# MediaPipe Setup
# =====================================

mp_hands = mp.solutions.hands

hands = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.3
)

results = hands.process(rgb)

# =====================================
# Prediction
# =====================================

if results.multi_hand_landmarks:

    landmarks = []

    for hand in results.multi_hand_landmarks:

        for lm in hand.landmark:

            landmarks.extend([
                lm.x,
                lm.y,
                lm.z
            ])

    X = np.array(
        landmarks
    ).reshape(1,-1)

    # Normalize
    X = (
        X - np.mean(X)
    ) / np.std(X)

    prediction = model.predict(
        X,
        verbose=0
    )

    confidence = np.max(
        prediction
    ) * 100

    predicted_index = np.argmax(
        prediction
    )

    predicted_label = encoder.inverse_transform(
        [predicted_index]
    )[0]

    if confidence < 75:

        print(
            "Unknown Sign"
        )

    else:

        print(
            "Prediction:",
            predicted_label
        )

        print(
            f"Confidence: {confidence:.2f}%"
        )

else:

    print(
        "No Hand Detected"
    )

Model Loaded
Label Encoder Loaded


Saving a.jpg to a (1).jpg


/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Prediction: N
Confidence: 99.99%


In [ ]:
import pickle

# Save label encoder again
with open(
    "label_encoder.pkl",
    "wb"
) as f:

    pickle.dump(
        encoder,
        f
    )

print("label_encoder.pkl Saved")

label_encoder.pkl Saved
